# Day 6: Layers — Organizing Neurons

**Learning Objective**: Implement `Layer` and `MLP` classes to organize neurons into a network.

A single neuron can only learn linear decision boundaries. By stacking neurons into **layers** and layers into **networks (MLPs)**, we can learn arbitrarily complex patterns.

In [ ]:
import math
import random
from graphviz import Digraph

## Theory

### Layer Architecture
A layer with 3 inputs and 4 outputs has 4 neurons, each receiving ALL inputs:
```
[x1, x2, x3] → [N1, N2, N3, N4] → [y1, y2, y3, y4]
```

### MLP Architecture
An MLP chains layers: `MLP(3, [4, 2])` means:
```
[3 inputs] → [4 neurons] → [2 neurons] → [2 outputs]
```

### Parameter Count
For MLP(3, [4, 2]):
- Layer 1: 4 neurons × (3 weights + 1 bias) = 16 params
- Layer 2: 2 neurons × (4 weights + 1 bias) = 10 params
- **Total: 26 parameters**

## Value and Neuron Classes (from Days 4 & 5)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return Value(other) * self**-1

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    
    def parameters(self):
        return self.w + [self.b]

## The Layer Class

A **Layer** is a collection of neurons that share the same input.

In [ ]:
class Layer:
    def __init__(self, nin, nout):
        """
        nin: number of inputs to each neuron
        nout: number of neurons in this layer
        """
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        """
        Forward pass: each neuron processes the same input.
        Returns single Value if nout=1, else list.
        """
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        """Return all parameters from all neurons."""
        return [p for n in self.neurons for p in n.parameters()]

## Test 1: Layer Forward Pass

In [ ]:
# Create a layer with 3 inputs and 4 neurons
layer = Layer(3, 4)

x = [Value(1.0), Value(2.0), Value(3.0)]
out = layer(x)

print(f"Output shape: {len(out)}")
print(f"Outputs: {[o.data for o in out]}")
print(f"Parameters: {len(layer.parameters())}")

# Expected: 4 neurons × (3 weights + 1 bias) = 16 params
assert len(out) == 4, "Should have 4 outputs"
assert len(layer.parameters()) == 16, "Should have 16 parameters"
print("\n✅ Layer works correctly!")

## The MLP Class

A **Multi-Layer Perceptron (MLP)** chains layers together sequentially.

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        """
        nin: number of inputs
        nouts: list of layer sizes [hidden1, hidden2, ..., output]
        """
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        """Forward pass through all layers."""
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        """Return all parameters from all layers."""
        return [p for layer in self.layers for p in layer.parameters()]

## Test 2: MLP Forward Pass

In [ ]:
# Create MLP: 3 inputs → 4 hidden → 4 hidden → 1 output
mlp = MLP(3, [4, 4, 1])

x = [Value(1.0), Value(2.0), Value(3.0)]
out = mlp(x)

print(f"Output: {out.data:.4f}")
print(f"Total parameters: {len(mlp.parameters())}")

# Parameter count:
# Layer 0: 4 × (3 + 1) = 16
# Layer 1: 4 × (4 + 1) = 20
# Layer 2: 1 × (4 + 1) = 5
# Total: 41

assert len(mlp.parameters()) == 41, f"Expected 41, got {len(mlp.parameters())}"
print("\n✅ MLP works correctly!")

## Test 3: Trace Forward Pass Layer by Layer

In [ ]:
def trace_mlp(mlp, x):
    """Print layer-by-layer activations."""
    print("=" * 50)
    print("MLP Forward Pass Trace")
    print("=" * 50)
    print(f"Input: {[xi.data for xi in x]}")
    
    for i, layer in enumerate(mlp.layers):
        x = layer(x)
        if isinstance(x, list):
            print(f"Layer {i} output: {[xi.data for xi in x]}")
        else:
            print(f"Layer {i} output: {x.data:.4f}")
    print("=" * 50)
    return x

mlp = MLP(2, [3, 1])
x = [Value(1.0), Value(-1.0)]
out = trace_mlp(mlp, x)

## Test 4: Gradient Flow Through MLP

In [ ]:
# Create small MLP
mlp = MLP(2, [4, 1])

# Forward pass
x = [Value(0.5), Value(-0.5)]
out = mlp(x)

print(f"Output: {out.data:.4f}")

# Backward pass
out.backward()

# Check all parameters have gradients
params = mlp.parameters()
non_zero_grads = sum(1 for p in params if p.grad != 0)

print(f"\nTotal parameters: {len(params)}")
print(f"Parameters with non-zero gradients: {non_zero_grads}")

# Show some gradients
print("\nFirst 5 parameter gradients:")
for i, p in enumerate(params[:5]):
    print(f"  param[{i}]: data={p.data:.4f}, grad={p.grad:.4f}")

print("\n✅ Gradients flow through entire network!")

## Test 5: Parameter Count Verification

In [ ]:
def verify_param_count(nin, nouts):
    """Calculate and verify parameter count."""
    mlp = MLP(nin, nouts)
    actual = len(mlp.parameters())
    
    # Calculate expected
    sz = [nin] + nouts
    expected = sum(sz[i+1] * (sz[i] + 1) for i in range(len(nouts)))
    
    print(f"MLP({nin}, {nouts})")
    print(f"  Expected: {expected} parameters")
    print(f"  Actual: {actual} parameters")
    assert actual == expected, "Parameter count mismatch!"
    print("  ✅ Verified!\n")

verify_param_count(3, [4, 2])      # 16 + 10 = 26
verify_param_count(2, [4, 4, 1])   # 12 + 20 + 5 = 37
verify_param_count(10, [16, 8, 1]) # 176 + 136 + 9 = 321

## Summary

Today we implemented:

1. **`Layer` class**:
   - Collection of neurons with shared input
   - Returns single Value when `nout=1`
   - Aggregates parameters from all neurons

2. **`MLP` class**:
   - Sequential layers with automatic dimension matching
   - Forward pass chains layer outputs to inputs
   - Aggregates parameters from all layers

**Key insight**: The network is now complete! We have:
- `Value` → automatic differentiation
- `Neuron` → single computational unit
- `Layer` → parallel neurons
- `MLP` → stacked layers

---

*Previous: [Day 5 — Neurons](./day_05_neurons.ipynb)*  
*Next: Day 7 — Training Loop*